# x2a_hprodsoj-hprodmil — Produtividade Histórica Pré-Painel de Soja e Milho (PAM/IBGE)

**Grupo no Modelo Econométrico:** `Geral`  
**Definição Operacional:** produtividade histórica de soja e milho, kg/ha, municipal, período anterior ao painel*  
**Conceito e Mecanismo Econômico:** Média de produtividade agrícola no período anterior a 2001. Atua como proxy de capacidade produtiva predeterminada, imune à simultaneidade contemporânea do desmatamento.  

---

### Origem dos Dados Brutos:
- `2_INTI\xdata\chap1\processed\tabular\h1_h2__pam_soja_milho__ibge__municipio_anual.csv`

### Observações e Regras Específicas do Documento de Hipóteses:
> - Alternativa a construir: média anterior a 2001, com janela e cobertura a definir.
> - Não preencher rendimento ausente com zero; avaliar a seleção dos municípios produtores.

### Salvaguarda Territorial (141 Municípios de Mato Grosso):
> **Regra Obrigatória do Projeto:** O estado de Mato Grosso é considerado com exatamente **141 municípios**.  
> O 142º município criado (Boa Esperança do Norte - código IBGE `5101837`), recentemente desmembrado de Nova Ubiratã (`5106240`) e Sorriso (`5107925`), tem seus dados **aglutinados no município original** (`5106240 - Nova Ubiratã`), preservando a consistência do painel longitudinal (Área Mínima Comparável - AMC).

---
## Protocolo Padronizado de 6 Fases (AED):
1. **Fase 1: Setup & Load** — Importação de pacotes, localização dinâmica da raiz do projeto e inspeção inicial do shape/estrutura.
2. **Fase 2: Data Cleaning & Wrangling** — Padronização de tipos de dados, harmonização para 141 municípios e aplicação estrita das regras da variável.
3. **Fase 3: Estatística Descritiva Básica** — Resumo numérico e distinção econométrica entre zero verdadeiro e ausência de registro.
4. **Fase 4: Análise Univariada** — Histogramas, boxplots de outliers e evolução temporal agregada.
5. **Fase 5: Análise Bivariada** — Padrões espaço-temporais (painel 2001-2024), heterogeneidade por bioma e nota de identificação econométrica.
6. **Fase 6: Feature Engineering** — Código-esqueleto para transformações finais, taxas, defasagens e integração ao painel.


## Fase 1: Setup & Load

Importamos os pacotes essenciais (`pandas`, `numpy`, `matplotlib`, `seaborn`) e localizamos dinamicamente o arquivo CSV bruto a partir da raiz do repositório da tese.

### Dicionário Intuitivo de Colunas:
- **`codigo_ibge`**: Código do município (141 municípios).
- **`ano`**: Ano (foco pré-2001).
- **`produto`**: Cultura agrícola.
- **`rendimento_medio_kg_ha`**: Produtividade média observada.


In [ ]:
# =============================================================================
# FASE 1: SETUP & LOAD
# =============================================================================
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuração visual para gráficos limpos e didáticos
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10

# Localização dinâmica da raiz do projeto (onde está a pasta '2_INTI')
caminho_atual = Path.cwd().resolve()
raiz_projeto = None
for p in [caminho_atual] + list(caminho_atual.parents):
    if (p / "2_INTI" / "RESEARCH_PROTOCOL.md").is_file() or (p / "AGENTS.md").is_file():
        raiz_projeto = p
        break

if raiz_projeto is None:
    raise FileNotFoundError("Não foi possível localizar a raiz do projeto (onde está 2_INTI/RESEARCH_PROTOCOL.md).")

# Caminho exato do arquivo conforme especificado no documento de hipóteses
caminho_csv = raiz_projeto / r"2_INTI\xdata\chap1\processed\tabular\h1_h2__pam_soja_milho__ibge__municipio_anual.csv"

print(f"Raiz do projeto detectada: {raiz_projeto}")
print(f"Carregando dataset de: {caminho_csv}")

if not caminho_csv.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {caminho_csv}")

# Leitura com suporte a caracteres especiais (BOM / UTF-8)
df = pd.read_csv(caminho_csv, sep=',', encoding='utf-8-sig', low_memory=False)

print(f"Dimensões do dataset (linhas, colunas): {df.shape}")
print("\n--- Primeiras 5 linhas do dataset bruto ---")
df.head()

In [ ]:
print("--- Informações estruturais das colunas e tipos de dados ---")
df.info()

## Fase 2: Data Cleaning & Wrangling

Realizamos a limpeza sistemática dos dados respeitando as regras específicas para `x2a_hprodsoj-hprodmil`:
- **Atenção:** Alternativa a construir: média anterior a 2001, com janela e cobertura a definir.
- **Atenção:** Não preencher rendimento ausente com zero; avaliar a seleção dos municípios produtores.
- **Salvaguarda Territorial:** Harmonização estrita para 141 municípios de Mato Grosso. Se houver dados de Boa Esperança do Norte (`5101837`), são aglutinados em Nova Ubiratã (`5106240`).


In [ ]:
# =============================================================================
# FASE 2: DATA CLEANING & WRANGLING
# =============================================================================
df.columns = df.columns.str.strip().str.lower().str.replace('\ufeff', '', regex=False)
df['codigo_ibge'] = df['codigo_ibge'].astype(str).str.zfill(7)
df['ano'] = pd.to_numeric(df['ano'], errors='coerce')
df['rendimento_medio_kg_ha'] = pd.to_numeric(df['rendimento_medio_kg_ha'], errors='coerce')

# HARMONIZAÇÃO TERRITORIAL (141 MUNICÍPIOS):
df['codigo_ibge'] = df['codigo_ibge'].replace({'5101837': '5106240'})

# REGRA DO TXT:
# 'Alternativa a construir: média anterior a 2001, com janela e cobertura a definir.'
# 'Não preencher rendimento ausente com zero; avaliar a seleção dos municípios produtores.'
df_pre = df[(df['codigo_ibge'].str.startswith('51')) & (df['ano'] < 2001)].copy()
print(f"Registros disponíveis no período pré-2001: {len(df_pre)}")

## Fase 3: Estatística Descritiva Básica

Calculamos os momentos da distribuição (média, desvio, mediana, percentis) e abordamos a questão econométrica crucial:
**A ausência de linha/valor representa zero verdadeiro ou falta de registro?**


In [ ]:
# =============================================================================
# FASE 3: ESTATÍSTICA DESCRITIVA BÁSICA
# =============================================================================
print("Produtividade histórica pré-2001 por cultura (kg/ha):")
print(df_pre.groupby('produto')['rendimento_medio_kg_ha'].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]))

## Fase 4: Análise Univariada

Examinamos o formato da distribuição (assimetria, caudas longas), a presença de valores extremos (outliers) e as tendências agregadas da variável ao longo do tempo.


In [ ]:
# =============================================================================
# FASE 4: ANÁLISE UNIVARIADA
# =============================================================================
sns.boxplot(data=df_pre.dropna(subset=['rendimento_medio_kg_ha']), x='produto', y='rendimento_medio_kg_ha')
plt.title("Boxplot da Produtividade Histórica Pré-2001")
plt.show()

## Fase 5: Análise Bivariada (Espaço-Temporal & Biomas)

Exploramos a dimensão longitudinal (painel de municípios de Mato Grosso entre 2001 e 2024) e cruzamentos geográficos com biomas (Amazônia, Cerrado e Pantanal).

> **Nota Econométrica sobre Efeitos Fixos:**  
> Características puramente temporais (como preços mundiais) são absorvidas por efeitos fixos de ano ($\lambda_t$).  
> Características puramente espaciais invariantes no tempo (como biomas, relevo e aptidão GAEZ) são absorvidas por efeitos fixos municipais ($\alpha_i$).  
> Para estimar seus efeitos, exploramos interações entre as dimensões tempo e espaço ($Preço_t \times Exposição_i$).


In [ ]:
# =============================================================================
# FASE 5: ANÁLISE BIVARIADA
# =============================================================================
cobertura = df_pre.groupby(['ano', 'produto'])['codigo_ibge'].nunique().reset_index()
sns.barplot(data=cobertura, x='ano', y='codigo_ibge', hue='produto')
plt.title("Número de Municípios com Produção Declarada no Período Pré-2001")
plt.ylabel("Qtd Municípios")
plt.show()

## Fase 6: Feature Engineering — Produtividade Observada e Rendimento Histórico

### 1. Step 1: Rendimento Médio Observado PAM/IBGE (kg/ha)
- **REGRA EXPRESSA DO DICIONÁRIO DE HIPÓTESES:** "Não preencher rendimento ausente com zero!"
- Se uma cidade não plantou soja em 2005, o rendimento é `NaN` (não observado), e NUNCA `0`. Se colocássemos zero, a econometria entenderia que os fazendeiros plantaram a lavoura inteira mas tiveram perda total de 100%!

### 2. Step 2: Produtividade Histórica Pré-Painel (Estrutural Invariante)
- **Média Pré-2001:** Calculamos a média do rendimento nos anos anteriores a 2001 ($ano \le 2000$) para cada município. Essa variável mede a fertilidade natural e capacidade histórica predeterminada, servindo como controle estático ou interação com preços.
- **Defasagem $t-1$ do Rendimento Contemporâneo:** Para mitigar endogeneidade e choques climáticos contemporâneos.


In [ ]:
# =============================================================================
# FASE 6: FEATURE ENGINEERING (TRATAMENTOS FINAIS E HARMONIZAÇÃO)
# =============================================================================
# 1. Separar rendimento observado em colunas por produto mantendo NaNs estritos
piv_rend = df.pivot_table(
    index=['codigo_ibge', 'ano'],
    columns='produto',
    values='rendimento_medio_kg_ha',
    aggfunc='mean'  # Não preencher com zero! Deixar ausente se não houve cultivo
).reset_index()

piv_rend.rename(
    columns={'Soja (em grão)': 'rend_soja_kg_ha', 'Milho (em grão)': 'rend_milho_kg_ha'},
    inplace=True
)

piv_rend.sort_values(['codigo_ibge', 'ano'], inplace=True)

# 2. Defasagem temporal de 1 ano (t-1)
# rend_soja_lag1: produtividade física observada da soja na safra anterior (kg/ha)
piv_rend['rend_soja_lag1'] = piv_rend.groupby('codigo_ibge')['rend_soja_kg_ha'].shift(1)
piv_rend['rend_milho_lag1'] = piv_rend.groupby('codigo_ibge')['rend_milho_kg_ha'].shift(1)

# 3. Construção da Produtividade Histórica Pré-2001 (Variável Estrutural Invariante)
# hprod_soja_pre2001: média histórica do rendimento nos anos 1990-2000 por município (kg/ha)
df_pre = df[df['ano'] <= 2000].groupby(['codigo_ibge', 'produto'])['rendimento_medio_kg_ha'].mean().unstack().reset_index()
df_pre.rename(
    columns={'Soja (em grão)': 'hprod_soja_pre2001', 'Milho (em grão)': 'hprod_milho_pre2001'},
    inplace=True
)

painel_rend = pd.merge(piv_rend, df_pre, on='codigo_ibge', how='left')
painel_rend = painel_rend[(painel_rend['ano'] >= 2001) & (painel_rend['ano'] <= 2024)].copy()

print("Painel de Rendimento Observado com salvaguarda de NaNs:")
print(f"Municípios com média histórica pré-2001 de soja calculada: {painel_rend['hprod_soja_pre2001'].notna().groupby(painel_rend['codigo_ibge']).any().sum()} / 141")
print(painel_rend[['codigo_ibge', 'ano', 'rend_soja_kg_ha', 'rend_soja_lag1', 'hprod_soja_pre2001']].head())